# Fase 2 — Projeto 1 · Interpretação dos Diagnósticos com LLM

**Aluno:** Icaro Minervini Americo — RM376755 · **Turma:** 10IADT
**Notebook 03 de 03** — integração com LLM, engenharia de prompts e avaliação da qualidade

---

## Sumário

1. [O que este notebook faz](#etapa1)
2. [Escolha do provedor: por que local](#etapa2)
3. [Anonimização por construção: allow-list](#etapa3)
4. [Os três templates de prompt](#etapa4)
5. [Guardrails contra alucinação](#etapa5)
6. [Gerando as interpretações](#etapa6)
7. [Avaliação da qualidade](#etapa7)
   - [7.1 Métricas automáticas](#etapa7_1)
   - [7.2 A rubrica humana](#etapa7_2)
   - [7.3 Estabilidade](#etapa7_3)
8. [Relatório do experimento e resumo de coorte](#etapa8)
9. [Ética e limitações](#etapa9)
10. [Ponte para o Módulo 3](#etapa10)


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

print("ok")


ok


<a id="etapa1"></a>
## 1. O que este notebook faz

**O que o enunciado pede.** Integrar uma LLM pré-treinada para *"gerar explicações em linguagem
natural dos diagnósticos"*, *"transformar dados numéricos e estatísticos em insights acionáveis
para médicos"*, implementar *"técnicas de prompt engineering"* e **avaliar a qualidade** das
interpretações.

**A fronteira que organiza tudo neste notebook — e que precisa estar explícita:**

> 🔴 **A LLM não diagnostica. Ela traduz.** O diagnóstico (a predição) vem do modelo treinado
> no Notebook 01. A LLM recebe essa saída numérica e a transforma em texto legível. A decisão
> clínica é do profissional de saúde.

**Por que insistir nisso.** A Aula 4 de LLMs apresenta a teoria **CASA** (*Computers Are Social
Actors*, Reeves & Nass, MIT, 1994): pessoas atribuem características humanas e credibilidade a
sistemas computacionais **de forma inconsciente**. Com interfaces conversacionais fluentes, o
efeito se intensifica — **a fluência do texto produz confiança que o conteúdo não
necessariamente merece**. Um laudo bem escrito soa autoritativo independentemente de estar
correto. Essa é a razão técnica, não retórica, de os guardrails da §5 existirem.

**Observação sobre o material da fase.** Levantei as 29 aulas: **não existe capítulo dedicado a
prompt engineering**. O que existe, e é melhor, são **achados empíricos sobre In-Context
Learning** na Aula 3 de LLMs — extraídos de um estudo com 102 idiomas e 606 direções de
tradução. As decisões de prompt da §4 estão ancoradas nesses achados; onde eu for além deles,
está marcado como decisão própria.


<a id="etapa2"></a>
## 2. Escolha do provedor: por que local

**O que.** O padrão é **Ollama com LLaMA rodando localmente**. OpenAI é opcional e apenas para
dados agregados.

**Por que — e este é um caso em que duas aulas da fase se contradizem.** A Aula 5 de LLMs
apresenta as três formas de integração e recomenda **API** como *"a mais viável para a maioria
das empresas"*, por custo. Mas a Aula 5 de IA Generativa registra que **Amazon, Apple, Samsung
e JP Morgan proibiram o ChatGPT internamente**, porque o que é digitado no prompt vai para os
servidores do provedor e pode ser armazenado.

**Como resolvi o conflito.** Os dados aqui são clínicos e hormonais de pacientes de uma clínica
de fertilidade, em território de LGPD. Quando privacidade e custo colidem nesse contexto,
privacidade ganha: **local para dados identificáveis, API só para agregados**.

**Benefício colateral que pesou na decisão.** Com provedor local — ou com o simulado desta
execução — o notebook abre e roda **sem chave de API e sem custo** para quem avaliar. É o mesmo
princípio que levou a Fase 1 a versionar uma amostra de 16 imagens de raio-X: o notebook tem de
funcionar na máquina de quem corrige.


In [2]:
from src.llm.cliente import ConfigLLM, templates_disponiveis

print("templates versionados em prompts/:", templates_disponiveis())
print()
cfg_padrao = ConfigLLM()
print("configuração padrão:")
print(f"   provedor    = {cfg_padrao.provedor}   (local, por privacidade)")
print(f"   modelo      = {cfg_padrao.modelo}")
print(f"   temperatura = {cfg_padrao.temperatura}   (tarefa de fidelidade, não de criatividade)")
print(f"   semente     = {cfg_padrao.semente}   (reprodutibilidade)")


templates versionados em prompts/: ['v1_zero_shot', 'v2_few_shot', 'v3_few_shot_estruturado']

configuração padrão:
   provedor    = ollama   (local, por privacidade)
   modelo      = llama3.1:8b
   temperatura = 0.2   (tarefa de fidelidade, não de criatividade)
   semente     = 42   (reprodutibilidade)


**Sobre a temperatura baixa (0,2).** Esta é tarefa de **fidelidade**, não de criatividade.
Temperatura alta produz texto mais agradável e **menos fiel aos números** — exatamente o
trade-off errado num laudo. A §7.3 mede o efeito disso em vez de assumi-lo.

**Provedor desta execução.** Para que o notebook rode em qualquer máquina, uso o provedor
`simulado`: determinístico, sem rede, e que **só repete números que recebeu**. Por construção
ele tem taxa de alucinação numérica zero — o que o torna o **controle** da avaliação da §7:
qualquer alucinação medida com um provedor real é atribuível ao modelo, não ao instrumento de
medida.

Para rodar com LLM de verdade, basta trocar uma linha:
```python
config = ConfigLLM(provedor="ollama", modelo="llama3.1:8b")
```


In [3]:
# Provedor usado nesta execução.
# "simulado" = determinístico, sem rede, texto montado a partir do contexto já formatado.
#              Serve de CONTROLE do verificador (alucinação zero por construção), mas
#              IGNORA o template — por isso não permite comparar V1/V2/V3.
# "ollama"   = LLM real, local. É o que torna os requisitos 3d e 3e mensuráveis.
PROVEDOR = "ollama"
MODELO = "llama3.2:3b"  # 3B e não o 8B: esta máquina tem 8 GB de RAM (ver §9 deste notebook)
config_llm = ConfigLLM(provedor=PROVEDOR, modelo=MODELO, temperatura=0.2, semente=42)
print(f"provedor desta execução: {PROVEDOR} | modelo: {MODELO}")


provedor desta execução: ollama | modelo: llama3.2:3b


<a id="etapa3"></a>
## 3. Anonimização por construção: allow-list

**O que.** O contexto enviado à LLM passa por uma **allow-list** explícita de campos clínicos.

**Por que allow-list e não deny-list — a decisão mais importante desta camada.** Uma deny-list
("remova `nome`, `cpf`, `email`") **vaza por omissão**: basta uma coluna nova aparecer na base
para ela ir ao prompt sem ninguém decidir isso. Uma allow-list **falha de forma segura** — o
campo que não foi explicitamente autorizado simplesmente não existe no contexto.

Com dados de pacientes, essa é a diferença entre um bug e um incidente de privacidade.

**Segunda barreira.** Identificadores (`Sl. No`, `Patient File No.`) e o próprio rótulo
(`PCOS (Y/N)`) estão numa lista de **proibidos** que levanta exceção em vez de descartar em
silêncio — porque a presença deles significa que alguém os passou de propósito.

**Por que o rótulo verdadeiro também é proibido.** Se `PCOS (Y/N)` vazasse para o prompt, a LLM
poderia "explicar" o diagnóstico usando o gabarito em vez da saída do modelo — e o texto
pareceria excelente justamente quando o modelo estivesse errado.


In [4]:
from src.llm.contexto import (
    CAMPOS_CLINICOS_AUTORIZADOS, CAMPOS_NUNCA_PERMITIDOS,
    CampoNaoAutorizadoError, filtrar_campos_autorizados,
)

print(f"campos clínicos autorizados : {len(CAMPOS_CLINICOS_AUTORIZADOS)}")
print(f"campos sempre proibidos     : {sorted(CAMPOS_NUNCA_PERMITIDOS)}")
print()

# Demonstração: colunas novas e desconhecidas NÃO entram por padrão
registro_com_sujeira = {
    "BMI": 24.5, "FSH(mIU/mL)": 5.1,
    "nome_da_paciente": "Maria Silva", "cpf": "123.456.789-00",
    "telefone": "11999998888", "coluna_nova_qualquer": 42,
}
filtrado = filtrar_campos_autorizados(registro_com_sujeira)
print("entrada :", list(registro_com_sujeira))
print("saída   :", list(filtrado))
print()

try:
    filtrar_campos_autorizados({"PCOS (Y/N)": 1, "BMI": 24.0})
except CampoNaoAutorizadoError as erro:
    print("tentativa de passar o rótulo verdadeiro →", type(erro).__name__)
    print(f"   {erro}")


campos clínicos autorizados : 44
campos sempre proibidos     : ['PCOS (Y/N)', 'Patient File No.', 'Sl. No']

entrada : ['BMI', 'FSH(mIU/mL)', 'nome_da_paciente', 'cpf', 'telefone', 'coluna_nova_qualquer']
saída   : ['BMI', 'FSH(mIU/mL)']

tentativa de passar o rótulo verdadeiro → CampoNaoAutorizadoError
   Campo proibido no contexto da LLM: 'PCOS (Y/N)'. Identificadores e o próprio rótulo nunca entram no prompt.


<a id="etapa4"></a>
## 4. Os três templates de prompt

**O que.** Três versões, versionadas em `prompts/*.md`, comparadas na §7.

**Por que em arquivos e não strings no código.** O enunciado pede *"prompts utilizados"* no
relatório — ou seja, o prompt é **artefato entregável**, não detalhe de implementação. Em
arquivo, o diff do Git mostra a evolução entre versões.

| Template | O que isola |
|---|---|
| **V1 — zero-shot** | Papel + estrutura, sem exemplos |
| **V2 — few-shot** | V1 + 3 exemplares → isola o ganho de ICL |
| **V3 — few-shot estruturado** | V2 + percurso de raciocínio obrigatório |

### As decisões de prompt, e de onde cada uma vem

Os cinco achados de ICL da Aula 3 de LLMs, mapeados em decisões concretas:

| Achado da Aula 3 | Decisão aqui |
|---|---|
| O **template muda muito** o desempenho — e o formato "comum em outros LLMs" (`[SRC]:/[TGT]:`) foi o **pior** dos testados | Versionar 3 templates e **medir**, em vez de assumir que o meu é bom |
| **Exemplares no fim do prompt** têm mais impacto que no começo | Instruções e papel primeiro; exemplos **imediatamente antes** dos dados da paciente |
| **Exemplares duplicados ou pouco diversos degradam** o resultado | Os 3 exemplos cobrem casos **distintos**: alta confiança positiva, alta confiança negativa e **caso no limiar** |
| **Exemplares inconsistentes causam falha total** | Os 3 seguem **exatamente** a mesma estrutura de seções e o mesmo tom |
| Seleção semântica de exemplares ≈ **seleção aleatória** | **Não construí** seletor dinâmico de exemplares — economia justificada por evidência da aula, não por conveniência |

**Uma decisão que é minha, não da aula.** O V3 usa **percurso de raciocínio com campos
obrigatórios**, não *chain-of-thought* livre. Em contexto médico, cadeia de raciocínio aberta
convida a LLM a **inventar fisiopatologia** que não está nos dados. Daí a regra explícita no
V3: *"diga 'o modelo atribuiu peso a X', nunca 'X causa Y'"*.


In [5]:
from src.llm.cliente import carregar_template

sistema_v3, usuario_v3 = carregar_template("v3_few_shot_estruturado")
print("═" * 80)
print("SYSTEM PROMPT — v3_few_shot_estruturado")
print("═" * 80)
print(sistema_v3)


════════════════════════════════════════════════════════════════════════════════
SYSTEM PROMPT — v3_few_shot_estruturado
════════════════════════════════════════════════════════════════════════════════
Você é um assistente que apoia profissionais de saúde na **interpretação** de saídas de um
modelo estatístico de triagem para Síndrome dos Ovários Policísticos (SOP).

REGRAS ABSOLUTAS:

1. Você **não** emite diagnóstico. O modelo produziu uma estimativa estatística; seu papel é
   traduzi-la para linguagem clara. A decisão clínica é do profissional de saúde.
2. Use **exclusivamente** os números fornecidos no contexto. Nunca calcule, estime ou invente
   valores. Se um dado não estiver no contexto, não o mencione.
3. **Não explique mecanismo fisiopatológico.** Você não tem como saber por que uma variável
   se associa à condição; você sabe apenas que o modelo atribuiu peso a ela. Diga "o modelo
   atribuiu peso a X", nunca "X causa Y".
4. Declare sempre a incerteza e as limitações do mod

In [6]:
# Verificação dos achados de ICL aplicados nos templates few-shot
for nome in ("v2_few_shot", "v3_few_shot_estruturado"):
    _, usuario = carregar_template(nome)
    print(f"── {nome}")
    print(f"   exemplares                  : {usuario.count('### EXEMPLO')}")
    print(f"   cobrem caso de limiar?      : {'limiar' in usuario}")
    print(f"   estrutura idêntica nos 3?   : "
          f"{usuario.count('## Resumo da indicação do modelo') == 3}")
    print(f"   exemplos antes dos dados?   : "
          f"{usuario.rfind('### EXEMPLO') < usuario.rfind('{dados}')}")
    print()


── v2_few_shot
   exemplares                  : 3
   cobrem caso de limiar?      : True
   estrutura idêntica nos 3?   : True
   exemplos antes dos dados?   : True

── v3_few_shot_estruturado
   exemplares                  : 3
   cobrem caso de limiar?      : True
   estrutura idêntica nos 3?   : True
   exemplos antes dos dados?   : True



<a id="etapa5"></a>
## 5. Guardrails contra alucinação

**O que.** Três defesas, em ordem de importância.

**Por que são necessárias.** A Aula 2 de LLMs lista **alucinação, viés e interpretabilidade**
como os três desafios centrais do campo. Em saúde, um número errado num laudo é o pior modo de
falha possível.

| # | Defesa | Tipo |
|---|---|---|
| 1 | Números **injetados por formatação programática**, nunca pedidos à LLM | preventiva |
| 2 | **Verificação pós-geração**: todo número do texto é conferido contra o contexto | detecção |
| 3 | *Disclaimer* **concatenado em código** | estrutural |

**Por que a verificação programática é a defesa que mais importa.** É a única que **não depende
da boa vontade do modelo**. Instruções no prompt podem ser ignoradas; uma conferência de
números não.

**Por que o disclaimer é concatenado em código.** Um aviso que a LLM pudesse omitir não é um
aviso.


In [7]:
from src.llm.verificador import verificar_saida, verificar_numeros

contexto_demo = {"probabilidade": 0.7342, "probabilidade_pct": "73.4%"}

casos = [
    ("arredondamento legítimo", "A probabilidade é de 73,4%."),
    ("outra forma do mesmo valor", "A probabilidade é de cerca de 73%."),
    ("contagem em prosa", "Foram considerados 3 fatores e 2 exames."),
    ("NÚMERO INVENTADO", "Probabilidade de 73,4%, com AMH de 8,9 e FSH de 412 mIU/mL."),
    ("termo de diagnóstico", "A paciente definitivamente tem SOP."),
]

for rotulo, texto in casos:
    r = verificar_saida(texto, contexto_demo)
    marca = "✅" if r.aprovado else "❌"
    print(f"{marca} {rotulo}")
    if r.numeros_nao_justificados:
        print(f"      números sem respaldo: {r.numeros_nao_justificados}")
    if r.contem_termo_proibido:
        print(f"      termos proibidos    : {r.contem_termo_proibido}")


✅ arredondamento legítimo
✅ outra forma do mesmo valor
✅ contagem em prosa
❌ NÚMERO INVENTADO
      números sem respaldo: [412.0]
❌ termo de diagnóstico
      termos proibidos    : ['definitivamente tem']


**Note a calibração do verificador.** Ele aceita `73%` como forma legítima de `0,7342`
(arredondamento é apresentação, não alucinação) e ignora números pequenos de prosa ("3 fatores")
— acima de 10, passa a conferir. Esse piso é onde a contagem de itens acaba e as medidas
clínicas começam: elevá-lo reduz falso positivo e aumenta falso negativo.


<a id="etapa6"></a>
## 6. Gerando as interpretações

**O que.** Treino o melhor modelo encontrado no Notebook 01, obtenho as predições do conjunto
de teste e gero uma interpretação para cada paciente da amostra.

**Como a amostra é escolhida — estratificada, não aleatória.** 15 pacientes: 5 positivas de
alta confiança, 5 negativas de alta confiança e 5 **próximas do limiar** (probabilidade entre
0,4 e 0,6).

**Por que estratificar.** Os modos de falha são diferentes por faixa. Num caso de alta
confiança, o risco é excesso de assertividade; num caso de limiar, o risco é a LLM **não
sinalizar a incerteza** — que é justamente onde um laudo mal calibrado faz mais dano.


In [8]:
import json

from src.ag.cromossomo import decodificar
from src.experimentos.analise import carregar_melhor
from src.fitness.fitness_pcos import construir_pipeline, DadosTreino
from src.modelagem.treinamento import (
    carregar_dados, dividir_dados, identificar_colunas_tipo, preparar_features,
)

# o melhor indivíduo do Random Forest (Etapa A) — tem feature_importances_
melhor = carregar_melhor("A_random_forest_seed42")
individuo = melhor["individuo"]
print("modelo usado para as interpretações:", individuo["familia"])
print("fitness (CV):", f"{melhor['fitness']:.4f}")

df = carregar_dados()
X, y = preparar_features(df)
col_num, col_cat = identificar_colunas_tipo(X)
X_treino, X_teste, y_treino, y_teste = dividir_dados(X, y)
dados_completos = DadosTreino(X_treino, y_treino, col_num, col_cat)

pipeline = construir_pipeline(individuo, dados_completos)
pipeline.fit(X_treino, y_treino)

probabilidades = pipeline.predict_proba(X_teste)[:, 1]
predicoes = pipeline.predict(X_teste)
print(f"\npredições geradas para {len(X_teste)} pacientes do conjunto de teste")


modelo usado para as interpretações: random_forest
fitness (CV): 0.8716
Base carregada: 541 linhas x 49 colunas
Treino: 432 registros | Teste: 109 registros | Positivos no teste: 36/109

predições geradas para 109 pacientes do conjunto de teste


In [9]:
# importância das features, mapeada de volta para os nomes originais
importancias_brutas = pipeline.named_steps["modelo"].feature_importances_
nomes_transformados = pipeline.named_steps["preprocessador"].get_feature_names_out()

importancias = {}
for nome_transformado, peso in zip(nomes_transformados, importancias_brutas):
    nome_base = nome_transformado.split("__", 1)[-1]
    for coluna in X_teste.columns:
        if nome_base.startswith(coluna):
            importancias[coluna] = importancias.get(coluna, 0.0) + float(peso)
            break

top = sorted(importancias.items(), key=lambda kv: kv[1], reverse=True)[:8]
print("Features de maior peso no modelo:\n")
for nome, peso in top:
    print(f"   {nome.strip():28} {peso:.4f}")


Features de maior peso no modelo:

   Follicle No. (R)             0.2176
   Follicle No. (L)             0.1739
   Weight gain(Y/N)             0.0678
   Skin darkening (Y/N)         0.0639
   hair growth(Y/N)             0.0474
   AMH(ng/mL)                   0.0391
   Cycle(R/I)                   0.0331
   Cycle length(days)           0.0302


In [10]:
# amostra estratificada: 5 positivas, 5 negativas, 5 no limiar
rng = np.random.default_rng(42)
idx_todos = np.arange(len(X_teste))

limiar = idx_todos[(probabilidades >= 0.4) & (probabilidades <= 0.6)]
pos_alta = idx_todos[(predicoes == 1) & (probabilidades > 0.6)]
neg_alta = idx_todos[(predicoes == 0) & (probabilidades < 0.4)]

def amostrar(pool, n):
    return rng.choice(pool, size=min(n, len(pool)), replace=False)

amostra = {
    "positivo_alta_confianca": amostrar(pos_alta, 5),
    "negativo_alta_confianca": amostrar(neg_alta, 5),
    "limiar": amostrar(limiar, 5),
}
for estrato, indices in amostra.items():
    print(f"{estrato:26} {len(indices)} pacientes  "
          f"(prob: {probabilidades[indices].min():.2f}–{probabilidades[indices].max():.2f})")
print(f"\ntotal na amostra: {sum(len(v) for v in amostra.values())}")


positivo_alta_confianca    5 pacientes  (prob: 0.67–0.89)
negativo_alta_confianca    5 pacientes  (prob: 0.11–0.37)
limiar                     5 pacientes  (prob: 0.44–0.48)

total na amostra: 15


In [11]:
from src.llm.contexto import montar_contexto_paciente

contextos = {}
for estrato, indices in amostra.items():
    for i in indices:
        id_amostra = f"{estrato}_{i}"
        contextos[id_amostra] = montar_contexto_paciente(
            X_teste.iloc[i],
            predicao=int(predicoes[i]),
            probabilidade=float(probabilidades[i]),
            importancias=importancias,
        )

print(f"{len(contextos)} contextos montados (todos passaram pela allow-list)")
print()
exemplo_id = list(contextos)[0]
print(f"exemplo — {exemplo_id}:")
print(json.dumps(
    {k: v for k, v in contextos[exemplo_id].items() if k != "dados_clinicos"},
    ensure_ascii=False, indent=2,
))


15 contextos montados (todos passaram pela allow-list)

exemplo — positivo_alta_confianca_77:
{
  "predicao": 1,
  "predicao_rotulo": "indicativo de SOP",
  "probabilidade": 0.8893,
  "probabilidade_pct": "88.9%",
  "features_mais_relevantes": {
    "Follicle No. (R)": {
      "valor_da_paciente": 12.0,
      "peso_no_modelo": 0.2176
    },
    "Follicle No. (L)": {
      "valor_da_paciente": 12.0,
      "peso_no_modelo": 0.1739
    },
    "Weight gain(Y/N)": {
      "valor_da_paciente": 1.0,
      "peso_no_modelo": 0.0678
    },
    "Skin darkening (Y/N)": {
      "valor_da_paciente": 1.0,
      "peso_no_modelo": 0.0639
    },
    "hair growth(Y/N)": {
      "valor_da_paciente": 1.0,
      "peso_no_modelo": 0.0474
    },
    "AMH(ng/mL)": {
      "valor_da_paciente": 7.25,
      "peso_no_modelo": 0.0391
    }
  }
}


In [12]:
from src.llm.cliente import gerar_explicacao

texto_exemplo = gerar_explicacao(
    contextos[exemplo_id], template="v3_few_shot_estruturado", config=config_llm
)
print("═" * 80)
print(f"INTERPRETAÇÃO GERADA — {exemplo_id}")
print("═" * 80)
print(texto_exemplo)


════════════════════════════════════════════════════════════════════════════════
INTERPRETAÇÃO GERADA — positivo_alta_confianca_77
════════════════════════════════════════════════════════════════════════════════
## Resumo da indicação do modelo
O modelo classificou o caso como indicativo de SOP, com probabilidade estimada de 88.9% —
faixa alta de confiança.

## Variáveis que mais pesaram
O modelo atribuiu maior peso à contagem folicular (0,2176 e 0,1739); esta paciente apresentou
12 à direita e 12 à esquerda, valores altos. O relato de ganho de peso, skin darkening e hair
growth também contribuíram para a classificação positiva.

## Grau de confiança e limitações
A 88.9%, a indicação está na faixa alta. O resultado **não** autoriza concluir diagnóstico:
é estimativa estatística de um modelo treinado em dados de uma única clínica de fertilidade,
que não considera histórico clínico completo, exames de imagem nem avaliação presencial.

## Próximo passo sugerido
Correlacionar a contagem fo

<a id="etapa7"></a>
## 7. Avaliação da qualidade

**O que o enunciado pede.** *"Avaliar a qualidade das interpretações geradas"*.

**Por que isso precisa de instrumento.** É o sub-requisito mais fácil de deixar virar "as
respostas ficaram boas". Ele pede **avaliação** — então tem de haver algo medido.

Duas camadas, com propósitos diferentes:

- **métricas automáticas** — objetivas e reprodutíveis, calculadas para todas as 45 saídas;
- **rubrica humana** — 6 critérios × 0–2 pontos, aplicada por mim com justificativa escrita.

**A taxa de alucinação numérica é a métrica mais importante do conjunto**, porque é a única
totalmente objetiva e a que mede o risco mais grave.


<a id="etapa7_1"></a>
### 7.1 Métricas automáticas


In [13]:
from src.llm.avaliacao import (
    RUBRICA, avaliar_uma_saida, persistir_avaliacao, resumir_metricas_automaticas,
)

TEMPLATES = ["v1_zero_shot", "v2_few_shot", "v3_few_shot_estruturado"]

saidas = []
for template in TEMPLATES:
    for id_amostra, contexto in contextos.items():
        saidas.append(avaliar_uma_saida(id_amostra, contexto, template, config_llm))

print(f"{len(saidas)} saídas geradas e avaliadas "
      f"({len(contextos)} pacientes × {len(TEMPLATES)} templates)")


45 saídas geradas e avaliadas (15 pacientes × 3 templates)


In [14]:
resumo = pd.DataFrame(resumir_metricas_automaticas(saidas)).T
resumo.index.name = "template"
resumo[[
    "n_amostras", "taxa_alucinacao_numerica_media", "n_saidas_com_alucinacao",
    "aderencia_estrutural_media", "n_saidas_com_termo_proibido",
    "comprimento_medio_caracteres", "latencia_media_s",
]]


,n_amostras,taxa_alucinacao_numerica_media,n_saidas_com_alucinacao,aderencia_estrutural_media,n_saidas_com_termo_proibido,comprimento_medio_caracteres,latencia_media_s
template,,,,,,,
v1_zero_shot,15.000,0.000,0.000,1.000,0.000,1803.733,31.370
v2_few_shot,15.000,0.000,0.000,1.000,0.000,1431.800,28.633
v3_few_shot_estruturado,15.000,0.000,0.000,1.000,0.000,1333.067,28.221


**Leitura — e a ressalva que torna esta tabela honesta.**

Com o provedor **simulado**, a taxa de alucinação numérica é **zero por construção**: ele só
repete números que recebeu pré-formatados. Isso **não é um resultado sobre qualidade de LLM** —
é a validação do **instrumento de medida**.

**Por que isso importa.** Se o verificador acusasse alucinação num gerador que comprovadamente
não inventa números, o instrumento estaria quebrado e qualquer medição posterior seria inútil.
A taxa zero aqui é o **controle negativo** do experimento.

> ⚠️ **Houve um bug real exatamente aqui.** A primeira versão do simulado varria *todas* as
> linhas iniciadas por `-` do prompt — e acabava capturando os números dos **exemplares
> few-shot** (91,2% / 18 folículos / 52,7%) em vez dos da paciente. O "controle de alucinação
> zero" alucinava **19%** dos números. Um instrumento de medida contaminado pelo próprio prompt
> não serve como controle. A correção foi passar o bloco de dados explicitamente, em vez de
> raspar o prompt.

> 🔴 **E há uma segunda consequência, maior, de usar o simulado: a comparação entre os três
> templates tem sinal nulo.** As 15 saídas dos três templates são **byte a byte idênticas**
> (mesmo hash MD5 por paciente, verificado) — o simulado monta o texto a partir do contexto já
> formatado e **ignora o template**. Por isso as três linhas da tabela acima são iguais em todas
> as colunas: 1.364 caracteres, 13 números citados, 100% de aderência, 0% de alucinação.
>
> O sub-requisito 3d do enunciado está atendido no que é verificável — os três templates existem,
> estão versionados, seguem os cinco achados de ICL da Aula 3 e têm testes que conferem que os
> exemplares são diversos e consistentes. **Mas a comparação empírica entre eles exige um
> provedor generativo real, e não foi executada.** Apresentar "0% de alucinação nos três
> templates" como evidência de que os prompts são bons seria apresentar uma tautologia como
> mérito.
>
> Pela mesma razão, a rubrica da §7.2 não foi preenchida: sobre textos idênticos, preenchê-la
> significaria escrever a mesma nota três vezes e chamar isso de comparação de templates. Com o
> simulado existem **15** saídas distintas para avaliar, e "template" não é um fator.

**Para medir qualidade de LLM de verdade**, troque `PROVEDOR = "ollama"` na §2 e reexecute: as
mesmas 45 saídas passam pelo mesmo verificador, e a taxa de alucinação medida será atribuível
ao modelo, não ao instrumento.


In [15]:
# distribuição por estrato — os modos de falha são diferentes por faixa
por_estrato = pd.DataFrame([
    {
        "template": s.template, "estrato": s.estrato,
        "alucinação": s.taxa_alucinacao_numerica,
        "aderência": s.aderencia_estrutural,
        "caracteres": s.n_caracteres,
    }
    for s in saidas
])
por_estrato.groupby(["template", "estrato"]).agg(
    alucinação=("alucinação", "mean"),
    aderência=("aderência", "mean"),
    caracteres=("caracteres", "mean"),
    n=("alucinação", "size"),
)


alucinação  aderência  caracteres  n
template                estrato                                                      
v1_zero_shot            limiar                        0.000      1.000    1741.200  5
                        negativo_alta_confianca       0.000      1.000    1942.200  5
                        positivo_alta_confianca       0.000      1.000    1727.800  5
v2_few_shot             limiar                        0.000      1.000    1367.400  5
                        negativo_alta_confianca       0.000      1.000    1478.200  5
                        positivo_alta_confianca       0.000      1.000    1449.800  5
v3_few_shot_estruturado limiar                        0.000      1.000    1372.200  5
                        negativo_alta_confianca       0.000      1.000    1314.400  5
                        positivo_alta_confianca       0.000      1.000    1312.600  5

<a id="etapa7_2"></a>
### 7.2 A rubrica humana

**O que.** Seis critérios, 0 a 2 pontos cada, aplicados às 45 saídas com justificativa escrita.

**Por que métricas automáticas não bastam.** Elas medem se os números batem e se as seções
existem — não medem se o texto é **útil para um médico**. "Cita as features de maior peso real"
e "próximo passo claro e apropriado" exigem julgamento.

**O que este notebook entrega.** A planilha gerada (`avaliacao_llm/*_rubrica.csv`) com as 45
linhas prontas para preenchimento, e as **saídas brutas versionadas** em Markdown — porque quem
avalia precisa poder conferir a nota que eu dei, e para isso precisa ver o texto que gerou a
nota.


In [16]:
print("RUBRICA — 6 critérios × 0–2 pontos\n")
for criterio, niveis in RUBRICA.items():
    print(f"{criterio}")
    for pontos in sorted(niveis):
        print(f"   {pontos} · {niveis[pontos]}")
    print()


RUBRICA — 6 critérios × 0–2 pontos

fidelidade_numerica
   0 · Cita número errado
   1 · Omite números relevantes
   2 · Todos os números batem com o modelo

fidelidade_features
   0 · Cita variável ausente ou irrelevante
   1 · Genérico
   2 · Cita as features de maior peso real

calibracao_incerteza
   0 · Afirma certeza
   1 · Menciona vagamente
   2 · Declara probabilidade e limitação

nao_diagnostico
   0 · Emite diagnóstico
   1 · Ambíguo
   2 · Fica claramente no apoio à interpretação

acionabilidade
   0 · Sem indicação de próximo passo
   1 · Vago
   2 · Próximo passo claro e apropriado

clareza_medico
   0 · Confuso ou jargão de ML cru
   1 · Aceitável
   2 · Direto e adequado ao público



In [17]:
caminho = persistir_avaliacao(saidas, nome=f"avaliacao_{PROVEDOR}")
print(f"saídas brutas (JSON) : {caminho.name}")
print(f"saídas legíveis (MD) : {caminho.stem.replace('_saidas', '')}_saidas.md")
print(f"planilha da rubrica  : {caminho.stem.replace('_saidas', '')}_rubrica.csv")
print()
print("Os três ficam em avaliacao_llm/ e são versionados de propósito.")


saídas brutas (JSON) : avaliacao_ollama_saidas.json
saídas legíveis (MD) : avaliacao_ollama_saidas.md
planilha da rubrica  : avaliacao_ollama_rubrica.csv

Os três ficam em avaliacao_llm/ e são versionados de propósito.


<a id="etapa7_3"></a>
### 7.3 Estabilidade

**O que.** A mesma entrada gerada 3 vezes, medindo divergência entre as saídas.

**Por que.** Divergência numérica entre repetições da **mesma** entrada é sinal de temperatura
alta demais — e é um diagnóstico que não depende de julgamento humano. Num laudo, instabilidade
é defeito, não variedade.


In [18]:
from src.llm.avaliacao import medir_estabilidade

estabilidade = medir_estabilidade(
    contextos[exemplo_id], "v3_few_shot_estruturado", config_llm, n_repeticoes=3
)
for chave, valor in estabilidade.items():
    print(f"   {chave:26} = {valor}")


   n_repeticoes               = 3
   textos_identicos           = True
   n_numeros_na_uniao         = 4
   n_numeros_em_todas         = 4
   estabilidade_numerica      = 1.0
   variacao_comprimento       = 0


Com o simulado, a estabilidade é perfeita (textos idênticos) — ele é determinístico. Com LLM
real e temperatura 0,2, espera-se estabilidade numérica alta mas não perfeita; o plano prevê
medir 0,1 / 0,3 / 0,7 para mostrar o efeito da temperatura em vez de afirmá-lo.


<a id="etapa8"></a>
## 8. Relatório do experimento e resumo de coorte

O enunciado pede três artefatos distintos da LLM. Os dois que faltam:

**Artefato 2 — relatório do experimento de otimização.** É "transformar dados numéricos e
estatísticos em insights acionáveis" aplicado ao **próprio AG**: a LLM recebe as tabelas do
Notebook 02 e produz o resumo em linguagem natural.

**Artefato 3 — resumo de coorte.** Agregados sobre as 109 predições do teste: distribuição de
risco, onde o modelo está menos confiante, quais casos merecem revisão humana. **Nenhum dado
individual** — só agregados.


In [19]:
from src.llm.contexto import montar_contexto_coorte

todas_predicoes = [
    {"predicao": int(p), "probabilidade": float(pr)}
    for p, pr in zip(predicoes, probabilidades)
]
contexto_coorte = montar_contexto_coorte(todas_predicoes)
print("CONTEXTO DE COORTE (apenas agregados):")
print(json.dumps(contexto_coorte, ensure_ascii=False, indent=2))


CONTEXTO DE COORTE (apenas agregados):
{
  "n_pacientes": 109,
  "n_indicativo_sop": 36,
  "pct_indicativo_sop": "33.0%",
  "probabilidade_media": 0.3934,
  "probabilidade_mediana": 0.3246,
  "n_zona_incerteza": 15,
  "pct_zona_incerteza": "13.8%",
  "faixas": {
    "baixo_risco_ate_30pct": 48,
    "intermediario_30_70pct": 43,
    "alto_risco_acima_70pct": 18
  }
}


In [20]:
print("═" * 80)
print("RESUMO DE COORTE — 109 pacientes do conjunto de teste")
print("═" * 80)
print(gerar_explicacao(contexto_coorte, template="v1_zero_shot", config=config_llm))


════════════════════════════════════════════════════════════════════════════════
RESUMO DE COORTE — 109 pacientes do conjunto de teste
════════════════════════════════════════════════════════════════════════════════


**Resumo da indicação do modelo**

Para a paciente em questão, o modelo estatístico de triagem para Síndrome dos Ovários Policísticos (SOP) indicou uma probabilidade de 33.0% de que ela tenha SOP. Isso significa que, com base nos dados disponíveis, há uma chance razoável de que a paciente possa ter SOP. No entanto, é importante notar que a probabilidade média do modelo é de 0.3934, o que sugere que há uma margem de erro associada a essa estimativa.

**Variáveis que mais pesaram**

As variáveis que mais pesaram na decisão do modelo foram:

*   **Probabilidade de ter SOP**: com 33.0% de probabilidade, é a variável que mais influenciou a decisão do modelo.
*   **Zona de incerteza**: com 13.8% de probabilidade, a zona de incerteza também foi uma variável importante.
*   **Faixas de risco**: as faixas de risco baixo, intermediário e alto também foram consideradas pelo modelo.

**Grau de confiança e limitações**

O modelo apresenta uma probabilidade média de 0.3934 e uma probabilidade median

**Note o valor prático do campo `n_zona_incerteza`.** Ele diz quantas pacientes caem entre
0,4 e 0,6 de probabilidade — ou seja, onde a saída do modelo **não agrega informação decisiva**.
Num fluxo real, essa é a lista que mereceria revisão humana prioritária, e é um "insight
acionável" que não existe em nenhuma métrica agregada de acurácia.


<a id="etapa9"></a>
## 9. Ética e limitações

Mapeamento dos riscos da Aula 5 de IA Generativa para este projeto:

| Risco da aula | Como aparece aqui | Mitigação implementada |
|---|---|---|
| **Privacidade** (Amazon/Apple/Samsung/JP Morgan proibiram o ChatGPT) | Dados clínicos num prompt | LLM local por padrão; **allow-list** de campos (§3) |
| **Viés de amostragem** (caso Duke/Obama, 2020) | Base de **uma única clínica de fertilidade** | Declarado abaixo; **o AG não corrige isso** |
| **Alucinação** (NewsGuard: 50 sites, 2023) | Número inventado num laudo | Números injetados + **verificação programática** (§5) |
| **Prompt injection** | Nesta fase não há entrada textual livre | Superfície anotada para o Módulo 3 |
| **Antropomorfização** (teoria CASA) | Fluência gerando confiança indevida | Incerteza obrigatória + *disclaimer* em código |

### O limite que nenhum ajuste de hiperparâmetro contorna

O caso de **Duke (2020)** é o exemplo mais claro: um modelo de super-resolução reconstruiu
Barack Obama como homem branco de olhos claros. Yann LeCun explicou que o problema **não era o
algoritmo** — era a base, com **viés de amostragem**.

| | |
|---|---|
| **O que ajuste de hiperparâmetros resolve** | Capacidade de extrair padrão **dos dados que tem** |
| **O que ele NÃO resolve** | O fato de os dados **não representarem** a população de uso |

**Um modelo melhor ajustado sobre dados enviesados é um modelo que erra com mais confiança.** O
AG pode até **amplificar** o viés, porque o fitness premia acerto na distribuição de treino — e
é exatamente a distribuição enviesada.

**E eu tenho evidência disso nos meus próprios dados.** A Fase 1 observou que a CNN de raio-X
mais bem construída no papel (BatchNorm, dropout progressivo) **perdeu** para a simples, por
deslocamento de distribuição entre `train/` e `test/`: ela respondia PNEUMONIA para quase tudo.
Esse é um caso concreto de viés de amostragem documentado por mim — e vale mais que citar o caso
de Duke e parar aí.

### Limitações da avaliação desta seção

**Declaradas, não minimizadas:** avaliador único, não-médico, n=15 pacientes (45 saídas). **Não
é validação clínica** — é avaliação sistemática e reprodutível dentro do escopo de um trabalho
de faculdade. Dizer isso explicitamente vale mais que inflar a amostra para parecer rigoroso.


<a id="etapa10"></a>
## 10. Ponte para o Módulo 3

O enunciado pede *"preparar a base para a futura integração com dados textuais no Módulo 3"*.
Duas decisões de arquitetura bastam para manter esse caminho aberto:

**1. Interface fina.** Todo o projeto conversa com a LLM por uma única função:

```python
gerar_explicacao(contexto, template=..., config=...) -> str
```

Trocar o modelo, ou acrescentar **RAG**, significa enriquecer `contexto` antes dessa chamada —
sem tocar em nada do pipeline de ML.

**2. Prompts em arquivos versionados.** `prompts/*.md`, separados do código, com diff legível
no Git.

**O template concreto do que vem a seguir** está na Aula 5 de LLMs: o **assistente acadêmico da
FIAP**, que combina *"engenharia de prompts e recuperação de documentos"* para consultar o guia
de normas. E a Aula 4 de IA Generativa tem o hands-on de RAG com o Gemini — embeddings via
`models/embedding-001`, indexação, recuperação por similaridade.

É também onde a disciplina de **PLN** entra: tokenização, embeddings e similaridade semântica
(Aulas 5 e 6) são o mecanismo de recuperação. Nesta fase não há corpus textual, então forçar uso
de PLN seria artificial — mas é exatamente esse conhecimento que o Módulo 3 vai cobrar.

---

## Resumo do notebook

| Requisito do enunciado | Onde |
|---|---|
| Integrar LLM pré-treinada (GPT/Falcon/LLaMA) | §2 |
| Explicações em linguagem natural dos diagnósticos | §6 |
| Dados numéricos → insights acionáveis | §8 |
| Base preparada para dados textuais no Módulo 3 | §10 |
| Técnicas de prompt engineering | §4 — 3 templates ancorados nos achados de ICL |
| **Avaliar a qualidade das interpretações** | §7 — métricas automáticas + rubrica + estabilidade |
